# ДЗ 10 · ERR-10 · Анализ ошибок и различия dev/test / Error analysis and distribution mismatch

**RU.** Реальные рукописные цифры UCI Optical Recognition of Handwritten Digits. Один объект — изображение 8×8, признаки — 64 счётчика пикселей 0…16, цель — цифра 0…9. Исходный официальный train содержит 3823 изображения от 30 авторов, test — 1797 изображений от других 13 авторов. В этой учебной задаче официальный train называется source, официальный test — target. Source делится на train/train-dev (80/20), target — на dev/test (50/50), seed=1010, stratify по цифре. Мы намеренно используем половину официального test для разработки и поэтому НЕ заявляем результат на стандартном benchmark. Разбиение внутри target не гарантирует новых авторов между dev/test: идентификаторов авторов в этих CSV нет. Все варианты модели получают одинаковые индексы. Test не участвует в выборе модели.

**EN.** Real UCI Optical Recognition of Handwritten Digits. Each observation is an 8×8 image, features are 64 pixel counts in 0…16, and the target is a digit 0…9. The official training file contains 3823 images from 30 writers; the official test file has 1797 images from 13 different writers. For this teaching protocol we call them source and target. Split source into train/train-dev (80/20), target into dev/test (50/50), seed=1010, stratified by digit. We deliberately repurpose half of the official test for development: this is NOT a standard benchmark score. Target dev/test are not guaranteed writer-disjoint because the CSV files have no writer IDs. All candidates use identical indices. Test is excluded from selection.

**Источник / Source:** [UCI, Alpaydin & Kaynak, Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), DOI 10.24432/C50P49, CC BY 4.0 (see source terms). Cite the dataset when reusing. CPU, NumPy, pandas, scikit-learn, matplotlib; the official ZIP is downloaded once and SHA-256 checked. No synthetic observations or labels. Historical sources: Archive/Tasks/Task4_ErrorEstimation.ipynb and Task5_Invariant.ipynb. Updated 2026: source/target protocol, explicit error tables, honest tests, no Google writes from notebooks.

**RU.** Четыре задания по 25 баллов. Тесты проверяют вычисления, а не обещанную победу модели. Итоговые штрафы определяет преподаватель по таблице сроков и практике №10. Заполните имя и группу, выполните Restart & Run All, сдайте .ipynb. Проверяющий повторяет запуск независимо. **EN.** Four 25-point tasks; tests assess calculations, not a promised model improvement. Instructor applies deadline/practice-10 policy. Fill your name/group, Restart & Run All, submit the notebook. The checker reruns independently.

In [ ]:
student_first = ""  # RU: имя; EN: first name.
student_last = ""  # RU: фамилия; EN: last name.
student_group = ""  # RU: группа; EN: group.
assignment_id = "ERR-10"  # RU: идентификатор ДЗ; EN: homework ID.

In [ ]:
import json, hashlib, zipfile, urllib.request  # RU: загружаем и проверяем архив; EN: download and verify the archive.
from pathlib import Path  # RU: задаём путь к кэшу; EN: define the cache path.
import numpy as np  # RU: работаем с пикселями и индексами; EN: manipulate pixels and indices.
import pandas as pd  # RU: строим таблицы ошибок; EN: construct error tables.
import matplotlib.pyplot as plt  # RU: показываем реальные изображения; EN: display real images.
from sklearn.model_selection import train_test_split  # RU: разделяем реальные наблюдения; EN: split real observations.
from sklearn.svm import SVC  # RU: обучаем компактный классификатор; EN: train a compact classifier.
from sklearn.metrics import confusion_matrix  # RU: независимо проверяем таблицу ошибок; EN: independently check confusion counts.
cache = Path("optdigits.zip")  # RU: повторно используем локальный архив; EN: reuse a local archive.
if not cache.exists():  # RU: скачиваем только при отсутствии; EN: download only when absent.
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/80/optical+recognition+of+handwritten+digits.zip", cache)  # RU: используем официальный UCI; EN: use the official UCI source.
assert hashlib.sha256(cache.read_bytes()).hexdigest() == "0d7b054fea010270e9b3f06411c654c5e59547732ad626381980baffe0a23fb0"  # RU: проверяем неизменность данных; EN: verify data integrity.
with zipfile.ZipFile(cache) as archive:  # RU: открываем CSV внутри ZIP без распаковки; EN: open CSV members without extraction.
    source = np.loadtxt(archive.open("optdigits.tra"), delimiter=",")  # RU: читаем исходных авторов; EN: read source writers.
    target = np.loadtxt(archive.open("optdigits.tes"), delimiter=",")  # RU: читаем других авторов; EN: read different target writers.
xs, ys = source[:, :64] / 16.0, source[:, 64].astype(int)  # RU: нормируем по известному максимуму; EN: normalize by the known maximum.
xt, yt = target[:, :64] / 16.0, target[:, 64].astype(int)  # RU: применяем то же преобразование к target; EN: apply the same transform to target.
train, train_dev = train_test_split(np.arange(len(ys)), test_size=0.2, stratify=ys, random_state=1010)  # RU: оставляем source-контроль; EN: hold out source validation.
dev, test = train_test_split(np.arange(len(yt)), test_size=0.5, stratify=yt, random_state=1010)  # RU: отделяем целевой test; EN: reserve target test.
assert not set(train) & set(train_dev) and not set(dev) & set(test)  # RU: запрещаем пересечение внутри источника; EN: disallow overlap within a source.
assert len(source) == 3823 and len(target) == 1797  # RU: проверяем полноту файлов; EN: check file completeness.
baseline = SVC(C=0.1, kernel="rbf", gamma="scale").fit(xs[train], ys[train])  # RU: fit видит только train; EN: fit sees training data only.
dev_pred = baseline.predict(xt[dev])  # RU: анализируем dev до выбора следующей модели; EN: inspect dev before selecting the next model.
print("train / train-dev / dev / test:", len(train), len(train_dev), len(dev), len(test))  # RU: выводим размеры разбиений; EN: display partition sizes.


## 1 · Матрица путаницы / Confusion counts

**RU.** Реализуйте `confusion_counts(truth, prediction)` без sklearn.metrics: целочисленная матрица 10×10, строка — истинная цифра, столбец — предсказанная. Сохраните нулевые строки отсутствующих классов. Примените к baseline dev и назовите наиболее частую внедиагональную пару. **EN.** Implement a 10×10 integer confusion matrix without sklearn.metrics, true digits on rows and predicted digits on columns, preserving absent classes. Apply to baseline dev and identify the largest off-diagonal pair.

In [ ]:
def confusion_counts(truth, prediction):  # RU: строим матрицу из пар меток; EN: build a matrix from label pairs.
    raise NotImplementedError("TODO / Ваш код")  # RU: реализуйте функцию; EN: implement the function.

## 2 · Поддержка и вклад / Support and contribution

**RU.** Реализуйте `slice_contribution(matrix)`: DataFrame digit/n/errors/rate/contribution; n — сумма строки, errors — вне диагонали, rate=errors/n (NaN при n=0), contribution=errors/N, где N — сумма ВСЕЙ матрицы. При N=0 вклад NaN. Покажите, что сумма вкладов равна общей ошибке; объясните отличие худшей доли ошибок от наибольшего вклада. **EN.** Return digit/n/errors/rate/contribution. n is row support; errors excludes the diagonal; rate=errors/n (NaN for n=0); contribution=errors/N for total N (NaN when N=0). Show that contributions sum to overall error and explain why highest rate need not mean largest contribution.

In [ ]:
def slice_contribution(matrix):  # RU: разделяем риск среза и вклад в общий итог; EN: separate slice risk from overall contribution.
    raise NotImplementedError("TODO / Ваш код")  # RU: реализуйте функцию; EN: implement the function.

## 3 · Доверительный интервал / Confidence interval

**RU.** Напишите `wilson_interval(errors,n,z=1.96)`: tuple(lower,upper) для доли ошибок. Формулы: p=errors/n; denominator=1+z²/n; center=(p+z²/(2n))/denominator; half=z*sqrt(p*(1-p)/n+z²/(4n²))/denominator. При n=0 верните (NaN,NaN), при errors<0 или errors>n выбросьте ValueError. Покажите интервалы для срезов dev. Это приближённые интервалы при независимости наблюдений; несколько изображений одного автора могут нарушать предположение. **EN.** Implement the specified Wilson interval, returning NaNs for zero support and ValueError for invalid counts. Display dev-slice intervals and explain the independent-observation assumption and writer correlation limitation.

In [ ]:
def wilson_interval(errors, n, z=1.96):  # RU: оцениваем неопределённость доли; EN: estimate rate uncertainty.
    raise NotImplementedError("TODO / Ваш код")  # RU: реализуйте функцию; EN: implement the function.

## 4 · Выбор с ограничением / Constrained selection

**RU.** Реализуйте `choose_model(records, max_source_error)`: допускайте только строки с train_dev_error<=max_source_error, затем выбирайте минимальный dev_error, при равенстве меньший C. Если допустимых нет — None. Используйте четыре кандидата практики и заранее заданное ограничение 0.05; возвращайте C. Затем зафиксируйте правило и один раз оцените test, отдельно напечатайте train/train-dev/dev/test. Напишите вывод RU/EN: что измерено, что лишь гипотеза, следующий эксперимент. **EN.** Keep candidates with train_dev_error<=max_source_error, minimize dev_error, break ties by smaller C, return C or None if infeasible. Use the four practice candidates and a predeclared 0.05 source-error limit. Freeze selection, evaluate test once, and separately report all four errors. Write RU/EN conclusions distinguishing evidence, hypothesis and next experiment.

In [ ]:
def choose_model(records, max_source_error):  # RU: выбираем при фиксированном ограничении; EN: select under a fixed constraint.
    raise NotImplementedError("TODO / Ваш код")  # RU: реализуйте функцию; EN: implement the function.

In [ ]:
score = 0

# Task 1: 25 points
assert np.array_equal(confusion_counts(yt[dev], dev_pred), confusion_matrix(yt[dev], dev_pred, labels=range(10)))
assert confusion_counts(yt[dev][:0], dev_pred[:0]).shape == (10, 10)
assert confusion_counts(yt[dev][:0], dev_pred[:0]).sum() == 0
score += 25


# Task 2: 25 points
cm_ref = confusion_matrix(yt[dev], dev_pred, labels=range(10))
sc = slice_contribution(cm_ref)
assert list(sc.columns) == ["digit", "n", "errors", "rate", "contribution"]
assert np.array_equal(sc["n"], cm_ref.sum(axis=1))
assert np.allclose(sc["rate"], (cm_ref.sum(axis=1)-np.diag(cm_ref))/cm_ref.sum(axis=1))
assert np.allclose(sc["contribution"], (cm_ref.sum(axis=1)-np.diag(cm_ref))/cm_ref.sum())
assert slice_contribution(confusion_matrix([], [], labels=range(10)))["rate"].isna().all()
score += 25


# Task 3: 25 points
from statistics import NormalDist
z = NormalDist().inv_cdf(0.975)
for row in sc.itertuples():
    lo, hi = wilson_interval(row.errors, row.n, z)
    p = row.errors/row.n
    expected_center = (row.errors+z*z/2)/(row.n+z*z)
    assert np.isclose((lo+hi)/2, expected_center)
    assert np.isclose((hi-lo)/2, z/(1+z*z/row.n)*np.sqrt(p*(1-p)/row.n+z*z/(4*row.n**2)))
    assert -1e-12 <= lo <= hi <= 1+1e-12
assert all(np.isnan(wilson_interval(0, 0)))
try:
    wilson_interval(2, 1)
except ValueError:
    pass
else:
    raise AssertionError("Reject impossible counts")
score += 25


# Task 4: 25 points
records = []
for c in [0.01, 0.1, 1, 10]:
    model = SVC(C=c, kernel="rbf", gamma="scale").fit(xs[train], ys[train])
    records.append({"C": c, "train_dev_error": float(np.mean(model.predict(xs[train_dev]) != ys[train_dev])), "dev_error": float(np.mean(model.predict(xt[dev]) != yt[dev]))})
for limit in [0.0, 0.05, 1.0]:
    eligible = sorted([r for r in records if r["train_dev_error"] <= limit], key=lambda r: (r["dev_error"], r["C"]))
    assert choose_model(records, limit) == (eligible[0]["C"] if eligible else None)
assert choose_model([], 0.05) is None
assert choose_model(list(reversed(records)), 0.05) == choose_model(records, 0.05)
score += 25

print(json.dumps({"name": (student_first+" "+student_last).strip(), "group": student_group, "assignment": assignment_id, "score": score, "max_score": 100}, ensure_ascii=False))